# Star Track 1: Distance-3 Surface Code Syndrome Extraction

In this notebook, we construct the stabilizer checks of a distance-3 surface code in Tsim and simulate two rounds of syndrome extraction.

The focus is on:
- identifying data and ancilla qubits,
- implementing Z- and X-type stabilizer measurements,
- repeating syndrome extraction over two rounds,
- and showing how Pauli errors appear as detection events.

For clarity and interpretability, Z- and X-type syndrome extraction are validated separately, and the final combined event vector is assembled from the two validated extractors.

In [1]:
from typing import Any
import numpy as np

from bloqade import squin, tsim
from bloqade.types import MeasurementResult, Qubit
from kirin.dialects.ilist import IList

Register = IList[Qubit, Any]
Measurement = IList[MeasurementResult, Any]

def show_circuit(squin_kernel):
    @squin.kernel
    def _to_visualize():
        _ = squin_kernel()
    return tsim.Circuit(_to_visualize).diagram(height=400)

## Distance-3 Surface Code Layout

We use a 3×3 data-qubit grid:

- Data qubits: `D0 ... D8`
- Z stabilizers:
  - `Z0 = D0 D1 D3 D4`
  - `Z1 = D1 D2 D4 D5`
  - `Z2 = D3 D4 D6 D7`
  - `Z3 = D4 D5 D7 D8`
- X stabilizers act on the same 2×2 blocks.

The stabilizer measurements use ancilla qubits to extract parity information without directly measuring the data qubits.

## 1. Single-Stabilizer Building Blocks

We first validate the two basic stabilizer measurement circuits:

- a Z stabilizer, which detects X errors,
- an X stabilizer, which detects Z errors.

These are the elementary building blocks of the full surface-code syndrome extraction circuit.

In [2]:
@squin.kernel
def z_check():
    q = squin.qalloc(5)   # q[0:4] data, q[4] ancilla

    squin.cx(q[0], q[4])
    squin.cx(q[1], q[4])
    squin.cx(q[2], q[4])
    squin.cx(q[3], q[4])

    squin.broadcast.measure([q[4]])

In [3]:
show_circuit(z_check)

In [4]:
samples = tsim.Circuit(z_check).compile_sampler().sample(shots=10)
print(samples.astype(int))

[[0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]]


In [5]:
@squin.kernel
def z_check_with_error():
    q = squin.qalloc(5)

    squin.x(q[0])  # inserted X error

    squin.cx(q[0], q[4])
    squin.cx(q[1], q[4])
    squin.cx(q[2], q[4])
    squin.cx(q[3], q[4])

    squin.broadcast.measure([q[4]])

In [6]:
show_circuit(z_check_with_error)

In [7]:
samples = tsim.Circuit(z_check_with_error).compile_sampler().sample(shots=10)
print(samples.astype(int))

[[1]
 [1]
 [1]
 [1]
 [1]
 [1]
 [1]
 [1]
 [1]
 [1]]


The Z stabilizer is deterministic on the initial `|0⟩` product state.  
After inserting a single X error on one data qubit, the stabilizer outcome flips, confirming that Z-type checks detect X-type errors.

In [8]:
@squin.kernel
def x_check():
    q = squin.qalloc(5)

    # prepare data qubits in |+>
    squin.h(q[0])
    squin.h(q[1])
    squin.h(q[2])
    squin.h(q[3])

    squin.h(q[4])
    squin.cx(q[4], q[0])
    squin.cx(q[4], q[1])
    squin.cx(q[4], q[2])
    squin.cx(q[4], q[3])
    squin.h(q[4])

    squin.broadcast.measure([q[4]])

In [9]:
show_circuit(x_check)

In [10]:

samples = tsim.Circuit(x_check).compile_sampler().sample(shots=10)
print(samples.astype(int))

[[0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]]


In [11]:
@squin.kernel
def x_check_with_error():
    q = squin.qalloc(5)

    # prepare data qubits in |+>
    squin.h(q[0])
    squin.h(q[1])
    squin.h(q[2])
    squin.h(q[3])

    squin.z(q[0])  # inserted Z error

    squin.h(q[4])
    squin.cx(q[4], q[0])
    squin.cx(q[4], q[1])
    squin.cx(q[4], q[2])
    squin.cx(q[4], q[3])
    squin.h(q[4])

    squin.broadcast.measure([q[4]])

In [12]:
samples = tsim.Circuit(x_check_with_error).compile_sampler().sample(shots=10)
print(samples.astype(int))

[[1]
 [1]
 [1]
 [1]
 [1]
 [1]
 [1]
 [1]
 [1]
 [1]]


For the X stabilizer test, the data qubits are prepared in `|+⟩` so that the X-type parity is deterministic.  
A single Z error flips the X stabilizer outcome, confirming that X-type checks detect Z-type errors.

## 2. Full Z-Stabilizer Round on the Distance-3 Layout

We now apply the four Z stabilizers across the full 3×3 data-qubit grid.  
This produces a 4-bit Z-type syndrome:
`[Z0, Z1, Z2, Z3]`.

In [13]:
@squin.kernel
def z_round():
    q = squin.qalloc(13)   # 9 data + 4 ancillas

    # Z0: D0 D1 D3 D4 -> q9
    squin.cx(q[0], q[9])
    squin.cx(q[1], q[9])
    squin.cx(q[3], q[9])
    squin.cx(q[4], q[9])

    # Z1: D1 D2 D4 D5 -> q10
    squin.cx(q[1], q[10])
    squin.cx(q[2], q[10])
    squin.cx(q[4], q[10])
    squin.cx(q[5], q[10])

    # Z2: D3 D4 D6 D7 -> q11
    squin.cx(q[3], q[11])
    squin.cx(q[4], q[11])
    squin.cx(q[6], q[11])
    squin.cx(q[7], q[11])

    # Z3: D4 D5 D7 D8 -> q12
    squin.cx(q[4], q[12])
    squin.cx(q[5], q[12])
    squin.cx(q[7], q[12])
    squin.cx(q[8], q[12])

    squin.broadcast.measure([q[9], q[10], q[11], q[12]])

In [14]:
show_circuit(z_round)

In [15]:

samples = tsim.Circuit(z_round).compile_sampler().sample(shots=10)
print(samples.astype(int))

[[0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]]


In [16]:
@squin.kernel
def z_round_center_error():
    q = squin.qalloc(13)

    squin.x(q[4])

    # Z0
    squin.cx(q[0], q[9])
    squin.cx(q[1], q[9])
    squin.cx(q[3], q[9])
    squin.cx(q[4], q[9])

    # Z1
    squin.cx(q[1], q[10])
    squin.cx(q[2], q[10])
    squin.cx(q[4], q[10])
    squin.cx(q[5], q[10])

    # Z2
    squin.cx(q[3], q[11])
    squin.cx(q[4], q[11])
    squin.cx(q[6], q[11])
    squin.cx(q[7], q[11])

    # Z3
    squin.cx(q[4], q[12])
    squin.cx(q[5], q[12])
    squin.cx(q[7], q[12])
    squin.cx(q[8], q[12])

    squin.broadcast.measure([q[9], q[10], q[11], q[12]])

In [17]:
show_circuit(z_round_center_error)

In [18]:
samples = tsim.Circuit(z_round_center_error).compile_sampler().sample(shots=10)
print(samples.astype(int))

[[1 1 1 1]
 [1 1 1 1]
 [1 1 1 1]
 [1 1 1 1]
 [1 1 1 1]
 [1 1 1 1]
 [1 1 1 1]
 [1 1 1 1]
 [1 1 1 1]
 [1 1 1 1]]


In [19]:
@squin.kernel
def z_round_error_q1():
    q = squin.qalloc(13)

    squin.x(q[1])  # off-center X error

    squin.cx(q[0], q[9])
    squin.cx(q[1], q[9])
    squin.cx(q[3], q[9])
    squin.cx(q[4], q[9])

    squin.cx(q[1], q[10])
    squin.cx(q[2], q[10])
    squin.cx(q[4], q[10])
    squin.cx(q[5], q[10])

    squin.cx(q[3], q[11])
    squin.cx(q[4], q[11])
    squin.cx(q[6], q[11])
    squin.cx(q[7], q[11])

    squin.cx(q[4], q[12])
    squin.cx(q[5], q[12])
    squin.cx(q[7], q[12])
    squin.cx(q[8], q[12])

    squin.broadcast.measure([q[9], q[10], q[11], q[12]])

In [20]:
show_circuit(z_round_error_q1)

In [21]:
samples = tsim.Circuit(z_round_error_q1).compile_sampler().sample(shots=10)
print(samples.astype(int))

[[1 1 0 0]
 [1 1 0 0]
 [1 1 0 0]
 [1 1 0 0]
 [1 1 0 0]
 [1 1 0 0]
 [1 1 0 0]
 [1 1 0 0]
 [1 1 0 0]
 [1 1 0 0]]


The error signatures are local:

- an X error on the center qubit flips all four neighboring Z stabilizers,
- an X error on an off-center qubit flips only the adjacent Z stabilizers.

This locality is the key idea behind surface-code decoding.

## 3. Full X-Stabilizer Round on the Distance-3 Layout

I next implement the four X stabilizers on the same 2×2 blocks.  
This produces a 4-bit X-type syndrome:
`[X0, X1, X2, X3]`.

In [22]:
@squin.kernel
def x_round():
    q = squin.qalloc(13)   # 9 data + 4 ancillas

    # prepare data in |+>
    squin.h(q[0]); squin.h(q[1]); squin.h(q[2])
    squin.h(q[3]); squin.h(q[4]); squin.h(q[5])
    squin.h(q[6]); squin.h(q[7]); squin.h(q[8])

    # X0: D0 D1 D3 D4 -> q9
    squin.h(q[9])
    squin.cx(q[9], q[0])
    squin.cx(q[9], q[1])
    squin.cx(q[9], q[3])
    squin.cx(q[9], q[4])
    squin.h(q[9])

    # X1: D1 D2 D4 D5 -> q10
    squin.h(q[10])
    squin.cx(q[10], q[1])
    squin.cx(q[10], q[2])
    squin.cx(q[10], q[4])
    squin.cx(q[10], q[5])
    squin.h(q[10])

    # X2: D3 D4 D6 D7 -> q11
    squin.h(q[11])
    squin.cx(q[11], q[3])
    squin.cx(q[11], q[4])
    squin.cx(q[11], q[6])
    squin.cx(q[11], q[7])
    squin.h(q[11])

    # X3: D4 D5 D7 D8 -> q12
    squin.h(q[12])
    squin.cx(q[12], q[4])
    squin.cx(q[12], q[5])
    squin.cx(q[12], q[7])
    squin.cx(q[12], q[8])
    squin.h(q[12])

    squin.broadcast.measure([q[9], q[10], q[11], q[12]])

In [23]:
samples = tsim.Circuit(x_round).compile_sampler().sample(shots=10)
print(samples.astype(int))

[[0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]]


In [24]:
@squin.kernel
def x_round_center_error():
    q = squin.qalloc(13)

    # prepare data in |+>
    squin.h(q[0]); squin.h(q[1]); squin.h(q[2])
    squin.h(q[3]); squin.h(q[4]); squin.h(q[5])
    squin.h(q[6]); squin.h(q[7]); squin.h(q[8])

    squin.z(q[4])  # center Z error

    squin.h(q[9])
    squin.cx(q[9], q[0]); squin.cx(q[9], q[1]); squin.cx(q[9], q[3]); squin.cx(q[9], q[4])
    squin.h(q[9])

    squin.h(q[10])
    squin.cx(q[10], q[1]); squin.cx(q[10], q[2]); squin.cx(q[10], q[4]); squin.cx(q[10], q[5])
    squin.h(q[10])

    squin.h(q[11])
    squin.cx(q[11], q[3]); squin.cx(q[11], q[4]); squin.cx(q[11], q[6]); squin.cx(q[11], q[7])
    squin.h(q[11])

    squin.h(q[12])
    squin.cx(q[12], q[4]); squin.cx(q[12], q[5]); squin.cx(q[12], q[7]); squin.cx(q[12], q[8])
    squin.h(q[12])

    squin.broadcast.measure([q[9], q[10], q[11], q[12]])

In [26]:
samples = tsim.Circuit(x_round_center_error).compile_sampler().sample(shots=10)
print(samples.astype(int))

[[1 1 1 1]
 [1 1 1 1]
 [1 1 1 1]
 [1 1 1 1]
 [1 1 1 1]
 [1 1 1 1]
 [1 1 1 1]
 [1 1 1 1]
 [1 1 1 1]
 [1 1 1 1]]


In [27]:
@squin.kernel
def x_round_error_q1():
    q = squin.qalloc(13)

    # prepare data in |+>
    squin.h(q[0]); squin.h(q[1]); squin.h(q[2])
    squin.h(q[3]); squin.h(q[4]); squin.h(q[5])
    squin.h(q[6]); squin.h(q[7]); squin.h(q[8])

    squin.z(q[1])  # off-center Z error

    squin.h(q[9])
    squin.cx(q[9], q[0]); squin.cx(q[9], q[1]); squin.cx(q[9], q[3]); squin.cx(q[9], q[4])
    squin.h(q[9])

    squin.h(q[10])
    squin.cx(q[10], q[1]); squin.cx(q[10], q[2]); squin.cx(q[10], q[4]); squin.cx(q[10], q[5])
    squin.h(q[10])

    squin.h(q[11])
    squin.cx(q[11], q[3]); squin.cx(q[11], q[4]); squin.cx(q[11], q[6]); squin.cx(q[11], q[7])
    squin.h(q[11])

    squin.h(q[12])
    squin.cx(q[12], q[4]); squin.cx(q[12], q[5]); squin.cx(q[12], q[7]); squin.cx(q[12], q[8])
    squin.h(q[12])

    squin.broadcast.measure([q[9], q[10], q[11], q[12]])

In [29]:
samples = tsim.Circuit(x_round_error_q1).compile_sampler().sample(shots=10)
print(samples.astype(int))

[[1 1 0 0]
 [1 1 0 0]
 [1 1 0 0]
 [1 1 0 0]
 [1 1 0 0]
 [1 1 0 0]
 [1 1 0 0]
 [1 1 0 0]
 [1 1 0 0]
 [1 1 0 0]]


The X-type syndrome behaves analogously:

- a Z error on the center qubit flips all four neighboring X stabilizers,
- a Z error on an off-center qubit flips only the adjacent X stabilizers.

Together, the Z- and X-type checks provide sensitivity to both bit-flip and phase-flip errors.

## 4. Two Rounds of Syndrome Extraction

Surface codes detect errors through **changes in syndrome over time**, not through absolute syndrome values alone.

For each stabilizer type, I simulate two consecutive rounds and define the detection event as:

`events = round2 XOR round1`

- no error between rounds → zero detection events,
- error inserted between rounds → nonzero detection events.

In [30]:
@squin.kernel
def two_round_z_only():
    q = squin.qalloc(17)
    # q[0:9]   -> data qubits
    # q[9:13]  -> round 1 Z ancillas
    # q[13:17] -> round 2 Z ancillas

    # ---------- Round 1 ----------
    squin.cx(q[0], q[9]);  squin.cx(q[1], q[9]);  squin.cx(q[3], q[9]);  squin.cx(q[4], q[9])
    squin.cx(q[1], q[10]); squin.cx(q[2], q[10]); squin.cx(q[4], q[10]); squin.cx(q[5], q[10])
    squin.cx(q[3], q[11]); squin.cx(q[4], q[11]); squin.cx(q[6], q[11]); squin.cx(q[7], q[11])
    squin.cx(q[4], q[12]); squin.cx(q[5], q[12]); squin.cx(q[7], q[12]); squin.cx(q[8], q[12])

    # ---------- Round 2 ----------
    squin.cx(q[0], q[13]); squin.cx(q[1], q[13]); squin.cx(q[3], q[13]); squin.cx(q[4], q[13])
    squin.cx(q[1], q[14]); squin.cx(q[2], q[14]); squin.cx(q[4], q[14]); squin.cx(q[5], q[14])
    squin.cx(q[3], q[15]); squin.cx(q[4], q[15]); squin.cx(q[6], q[15]); squin.cx(q[7], q[15])
    squin.cx(q[4], q[16]); squin.cx(q[5], q[16]); squin.cx(q[7], q[16]); squin.cx(q[8], q[16])

    squin.broadcast.measure([q[9], q[10], q[11], q[12], q[13], q[14], q[15], q[16]])

In [31]:
@squin.kernel
def two_round_z_only_center_x_error():
    q = squin.qalloc(17)

    # ---------- Round 1 ----------
    squin.cx(q[0], q[9]);  squin.cx(q[1], q[9]);  squin.cx(q[3], q[9]);  squin.cx(q[4], q[9])
    squin.cx(q[1], q[10]); squin.cx(q[2], q[10]); squin.cx(q[4], q[10]); squin.cx(q[5], q[10])
    squin.cx(q[3], q[11]); squin.cx(q[4], q[11]); squin.cx(q[6], q[11]); squin.cx(q[7], q[11])
    squin.cx(q[4], q[12]); squin.cx(q[5], q[12]); squin.cx(q[7], q[12]); squin.cx(q[8], q[12])

    # inserted X error between rounds
    squin.x(q[4])

    # ---------- Round 2 ----------
    squin.cx(q[0], q[13]); squin.cx(q[1], q[13]); squin.cx(q[3], q[13]); squin.cx(q[4], q[13])
    squin.cx(q[1], q[14]); squin.cx(q[2], q[14]); squin.cx(q[4], q[14]); squin.cx(q[5], q[14])
    squin.cx(q[3], q[15]); squin.cx(q[4], q[15]); squin.cx(q[6], q[15]); squin.cx(q[7], q[15])
    squin.cx(q[4], q[16]); squin.cx(q[5], q[16]); squin.cx(q[7], q[16]); squin.cx(q[8], q[16])

    squin.broadcast.measure([q[9], q[10], q[11], q[12], q[13], q[14], q[15], q[16]])

In [32]:
@squin.kernel
def two_round_x_only():
    q = squin.qalloc(17)

    # prepare data in |+>
    squin.h(q[0]); squin.h(q[1]); squin.h(q[2])
    squin.h(q[3]); squin.h(q[4]); squin.h(q[5])
    squin.h(q[6]); squin.h(q[7]); squin.h(q[8])

    # ---------- Round 1 ----------
    squin.h(q[9]);  squin.cx(q[9], q[0]);  squin.cx(q[9], q[1]);  squin.cx(q[9], q[3]);  squin.cx(q[9], q[4]);  squin.h(q[9])
    squin.h(q[10]); squin.cx(q[10], q[1]); squin.cx(q[10], q[2]); squin.cx(q[10], q[4]); squin.cx(q[10], q[5]); squin.h(q[10])
    squin.h(q[11]); squin.cx(q[11], q[3]); squin.cx(q[11], q[4]); squin.cx(q[11], q[6]); squin.cx(q[11], q[7]); squin.h(q[11])
    squin.h(q[12]); squin.cx(q[12], q[4]); squin.cx(q[12], q[5]); squin.cx(q[12], q[7]); squin.cx(q[12], q[8]); squin.h(q[12])

    # ---------- Round 2 ----------
    squin.h(q[13]); squin.cx(q[13], q[0]); squin.cx(q[13], q[1]); squin.cx(q[13], q[3]); squin.cx(q[13], q[4]); squin.h(q[13])
    squin.h(q[14]); squin.cx(q[14], q[1]); squin.cx(q[14], q[2]); squin.cx(q[14], q[4]); squin.cx(q[14], q[5]); squin.h(q[14])
    squin.h(q[15]); squin.cx(q[15], q[3]); squin.cx(q[15], q[4]); squin.cx(q[15], q[6]); squin.cx(q[15], q[7]); squin.h(q[15])
    squin.h(q[16]); squin.cx(q[16], q[4]); squin.cx(q[16], q[5]); squin.cx(q[16], q[7]); squin.cx(q[16], q[8]); squin.h(q[16])

    squin.broadcast.measure([q[9], q[10], q[11], q[12], q[13], q[14], q[15], q[16]])

In [33]:
@squin.kernel
def two_round_x_only_center_z_error():
    q = squin.qalloc(17)

    # prepare data in |+>
    squin.h(q[0]); squin.h(q[1]); squin.h(q[2])
    squin.h(q[3]); squin.h(q[4]); squin.h(q[5])
    squin.h(q[6]); squin.h(q[7]); squin.h(q[8])

    # ---------- Round 1 ----------
    squin.h(q[9]);  squin.cx(q[9], q[0]);  squin.cx(q[9], q[1]);  squin.cx(q[9], q[3]);  squin.cx(q[9], q[4]);  squin.h(q[9])
    squin.h(q[10]); squin.cx(q[10], q[1]); squin.cx(q[10], q[2]); squin.cx(q[10], q[4]); squin.cx(q[10], q[5]); squin.h(q[10])
    squin.h(q[11]); squin.cx(q[11], q[3]); squin.cx(q[11], q[4]); squin.cx(q[11], q[6]); squin.cx(q[11], q[7]); squin.h(q[11])
    squin.h(q[12]); squin.cx(q[12], q[4]); squin.cx(q[12], q[5]); squin.cx(q[12], q[7]); squin.cx(q[12], q[8]); squin.h(q[12])

    # inserted Z error between rounds
    squin.z(q[4])

    # ---------- Round 2 ----------
    squin.h(q[13]); squin.cx(q[13], q[0]); squin.cx(q[13], q[1]); squin.cx(q[13], q[3]); squin.cx(q[13], q[4]); squin.h(q[13])
    squin.h(q[14]); squin.cx(q[14], q[1]); squin.cx(q[14], q[2]); squin.cx(q[14], q[4]); squin.cx(q[14], q[5]); squin.h(q[14])
    squin.h(q[15]); squin.cx(q[15], q[3]); squin.cx(q[15], q[4]); squin.cx(q[15], q[6]); squin.cx(q[15], q[7]); squin.h(q[15])
    squin.h(q[16]); squin.cx(q[16], q[4]); squin.cx(q[16], q[5]); squin.cx(q[16], q[7]); squin.cx(q[16], q[8]); squin.h(q[16])

    squin.broadcast.measure([q[9], q[10], q[11], q[12], q[13], q[14], q[15], q[16]])

In [34]:
def get_events(kernel):
    samples = tsim.Circuit(kernel).compile_sampler().sample(shots=10)
    round1 = samples[:, :4]
    round2 = samples[:, 4:]
    events = round1 ^ round2
    return round1.astype(int), round2.astype(int), events.astype(int)

In [35]:
z_r1, z_r2, z_events = get_events(two_round_z_only)
x_r1, x_r2, x_events = get_events(two_round_x_only)

print("Z events (no error):")
print(z_events)

print("\nX events (no error):")
print(x_events)

combined_no_error = np.concatenate([z_events, x_events], axis=1)
print("\nCombined events [Z0 Z1 Z2 Z3 X0 X1 X2 X3] (no error):")
print(combined_no_error)

Z events (no error):
[[0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]]

X events (no error):
[[0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]]

Combined events [Z0 Z1 Z2 Z3 X0 X1 X2 X3] (no error):
[[0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]]


In [36]:
z_r1, z_r2, z_events = get_events(two_round_z_only_center_x_error)
x_r1, x_r2, x_events = get_events(two_round_x_only_center_z_error)

print("Z events after center X error:")
print(z_events)

print("\nX events after center Z error:")
print(x_events)

combined_center_error = np.concatenate([z_events, x_events], axis=1)
print("\nCombined events [Z0 Z1 Z2 Z3 X0 X1 X2 X3] with injected center errors:")
print(combined_center_error)

Z events after center X error:
[[1 1 1 1]
 [1 1 1 1]
 [1 1 1 1]
 [1 1 1 1]
 [1 1 1 1]
 [1 1 1 1]
 [1 1 1 1]
 [1 1 1 1]
 [1 1 1 1]
 [1 1 1 1]]

X events after center Z error:
[[1 1 1 1]
 [1 1 1 1]
 [1 1 1 1]
 [1 1 1 1]
 [1 1 1 1]
 [1 1 1 1]
 [1 1 1 1]
 [1 1 1 1]
 [1 1 1 1]
 [1 1 1 1]]

Combined events [Z0 Z1 Z2 Z3 X0 X1 X2 X3] with injected center errors:
[[1 1 1 1 1 1 1 1]
 [1 1 1 1 1 1 1 1]
 [1 1 1 1 1 1 1 1]
 [1 1 1 1 1 1 1 1]
 [1 1 1 1 1 1 1 1]
 [1 1 1 1 1 1 1 1]
 [1 1 1 1 1 1 1 1]
 [1 1 1 1 1 1 1 1]
 [1 1 1 1 1 1 1 1]
 [1 1 1 1 1 1 1 1]]


## Conclusion

This notebook constructs the stabilizer structure of a distance-3 surface code in Tsim and demonstrates two rounds of syndrome extraction.

Main observations:
- Z stabilizers detect X errors.
- X stabilizers detect Z errors.
- Error location is encoded in the local syndrome pattern.
- Repeated rounds allow errors to be identified through detection events:
  `round2 XOR round1`.

For clarity, Z- and X-type syndrome extraction were validated separately, and the final 8-bit combined detection-event vector was assembled from the two validated extractors.